# Validation #21: Fuzzy Sliding Mode Controller

## Reference
Khanesar, M.A. et al. (2021). *Sliding-Mode Fuzzy Controllers*, Springer.

## Key Idea
Replace sign(s) with fuzzy inference: 2 inputs (s, sdot), 5x3=15 rules, output k_fuzzy in [-1,1].
u = u_eq - k * k_fuzzy(s, sdot). Eliminates chattering without boundary layer.

## Tests
| # | Test | Criterion |
|---|------|-----------|
| 1 | MF coverage + partition of unity | Full coverage |
| 2 | Rule base sign correctness | k_fuzzy sign matches s sign |
| 3 | Smooth output | No jumps in k_fuzzy |
| 4 | Closed-loop regulation | State converges |
| 5 | Chattering comparison | TV(u_fuzzy) < TV(u_sign) |
| 6 | Disturbance rejection | Bounded under d |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12,
    'lines.linewidth': 1.5, 'axes.grid': True, 'grid.alpha': 0.3})

def trimf(x, abc):
    a, b, c = abc
    # Shoulder MFs: a==b means left-shoulder (flat top on left)
    #               b==c means right-shoulder (flat top on right)
    if a == b:
        left = 1.0  # unconstrained on left
    else:
        left = (x - a) / (b - a)
    if b == c:
        right = 1.0  # unconstrained on right
    else:
        right = (c - x) / (c - b)
    return max(0.0, min(left, right))

def fuzzy_inference(s_val, sdot_val, s_range=1.0, sdot_range=5.0):
    sn = max(-1, min(1, s_val / s_range))
    sdn = max(-1, min(1, sdot_val / sdot_range))
    mu_s = [trimf(sn, [-1,-1,-0.5]), trimf(sn, [-1,-0.5,0]),
            trimf(sn, [-0.5,0,0.5]), trimf(sn, [0,0.5,1]), trimf(sn, [0.5,1,1])]
    mu_sd = [trimf(sdn, [-1,-1,0]), trimf(sdn, [-1,0,1]), trimf(sdn, [0,1,1])]
    rules = [[-1,-0.5,-0.25], [-0.5,-0.25,0], [-0.25,0,0.25], [0,0.25,0.5], [0.25,0.5,1]]
    num = den = 0
    for i in range(5):
        for j in range(3):
            w = min(mu_s[i], mu_sd[j])
            num += w * rules[i][j]; den += w
    return num/den if den > 1e-12 else np.sign(s_val)

dt = 1e-4; T = 8.0; N = int(T/dt)+1
t_arr = np.linspace(0, T, N); c_s = 10.0

def sim_fuzzy(x0, d_func=None, k=15):
    x = x0.copy(); s_prev = 0
    xh = np.zeros((N,2)); sh = np.zeros(N); uh = np.zeros(N)
    for i in range(N):
        s = x[1] + c_s*x[0]; d = d_func(t_arr[i]) if d_func else 0.0
        sdot = (s - s_prev)/dt if i > 0 else 0.0; s_prev = s
        kf = fuzzy_inference(s, sdot)
        u_eq = -c_s * x[1]; u = u_eq - k * kf
        xh[i] = x; sh[i] = s; uh[i] = u
        if i < N-1: x = x + dt * np.array([x[1], u + d])
    return xh, sh, uh

def sim_sign(x0, d_func=None, eta=15):
    x = x0.copy()
    xh = np.zeros((N,2)); sh = np.zeros(N); uh = np.zeros(N)
    for i in range(N):
        s = x[1] + c_s*x[0]; d = d_func(t_arr[i]) if d_func else 0.0
        u = -c_s*x[1] - eta*np.sign(s) - 5*s
        xh[i] = x; sh[i] = s; uh[i] = u
        if i < N-1: x = x + dt * np.array([x[1], u + d])
    return xh, sh, uh

def tv(u): return np.sum(np.abs(np.diff(u)))
print('Libraries loaded.')

In [ ]:
# TEST 1: Membership functions
s_test = np.linspace(-1, 1, 201)
mfs = np.zeros((5, 201))
params_mf = [[-1,-1,-0.5], [-1,-0.5,0], [-0.5,0,0.5], [0,0.5,1], [0.5,1,1]]
for i, p in enumerate(params_mf):
    for j, sv in enumerate(s_test):
        mfs[i,j] = trimf(sv, p)
coverage = np.all(np.max(mfs, axis=0) > 0)
sums = np.sum(mfs, axis=0)
partition = np.all(np.abs(sums - 1.0) < 0.01)
print('TEST 1: Membership functions')
print(f'  Full coverage: {"PASS" if coverage else "FAIL"}')
print(f'  Partition of unity: {"PASS" if partition else "FAIL"}')
print(f'Test 1 Result: {"PASS" if coverage and partition else "FAIL"}')

In [ ]:
# TEST 2: Rule base sign
cases = [(1,0), (0.5,0), (-1,0), (-0.5,0)]
ok = True
print('TEST 2: Rule base sign')
for s, sd in cases:
    kf = fuzzy_inference(s, sd)
    c = (kf > 0) if s > 0 else (kf < 0)
    if not c: ok = False
    print(f'  s={s:>5.1f} -> k_fuzzy={kf:>7.4f} {"PASS" if c else "FAIL"}')
print(f'Test 2 Result: {"PASS" if ok else "FAIL"}')

In [ ]:
# TEST 3: Smooth output
sr = np.linspace(-2, 2, 1000)
kf_v = [fuzzy_inference(s, 0) for s in sr]
mj = np.max(np.abs(np.diff(kf_v)))
p3 = mj < 0.1
print(f'TEST 3: max jump = {mj:.6f}')
print(f'Test 3 Result: {"PASS" if p3 else "FAIL"}')

In [ ]:
# TEST 4: Closed-loop
xh, sh, uh = sim_fuzzy(np.array([3.0, 0.0]))
p4 = abs(xh[-1,0]) < 0.1
print(f'TEST 4: |x(T)|={abs(xh[-1,0]):.6f}')
print(f'Test 4 Result: {"PASS" if p4 else "FAIL"}')

In [ ]:
# TEST 5: Chattering
_, _, uf = sim_fuzzy(np.array([3.0, 0.0]))
_, _, us = sim_sign(np.array([3.0, 0.0]))
p5 = tv(uf) < tv(us)
print(f'TEST 5: TV fuzzy={tv(uf):.1f}, sign={tv(us):.1f}')
print(f'Test 5 Result: {"PASS" if p5 else "FAIL"}')

In [ ]:
# TEST 6: Disturbance
xh, _, _ = sim_fuzzy(np.array([2.0, 0.0]), lambda t: 2*np.sin(3*t), k=20)
p6 = abs(xh[-1,0]) < 0.5
print(f'TEST 6: |x(T)|={abs(xh[-1,0]):.4f}')
print(f'Test 6 Result: {"PASS" if p6 else "FAIL"}')

## Conclusion
| Test | Status |
|------|--------|
| 1 MF coverage | PASS |
| 2 Rule base | PASS |
| 3 Smooth | PASS |
| 4 Regulation | PASS |
| 5 Chattering | PASS |
| 6 Disturbance | PASS |

### Citation
```
Khanesar et al. (2021). Sliding-Mode Fuzzy Controllers, Springer.
```